In [ ]:
# ========================= ZEpto RAG - COMPLETE JUPYTER VERSION =========================


import os
import json
from pathlib import Path
from typing import TypedDict, Literal, Any

import chromadb
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

# Optional OpenAI package
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

print("=" * 70)
print("ZEpto POLICY RAG - JUPYTER NOTEBOOK")
print("=" * 70)

# -------------------------------------------------------------------
# 1. JUPYTER-SAFE PATH CONFIGURATION
# -------------------------------------------------------------------

BASE_DIR = Path.cwd()
DOCS_DIR = BASE_DIR / "docs"
CHROMA_DIR = BASE_DIR / "chroma_db"

DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE_DIR)
print("Documents directory:", DOCS_DIR)

# -------------------------------------------------------------------
# 2. CREATE ALL 8 POLICY DOCUMENTS
# -------------------------------------------------------------------

documents = {
    "doc_01.txt": """Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.""",

    "doc_02.txt": """Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.""",

    "doc_03.txt": """Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.""",

    "doc_04.txt": """Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.""",

    "doc_05.txt": """Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.""",

    "doc_06.txt": """If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.""",

    "doc_07.txt": """Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.""",

    "doc_08.txt": """Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."""
}

for filename, content in documents.items():
    (DOCS_DIR / filename).write_text(content, encoding="utf-8")

print("Created 8 policy documents.")

# -------------------------------------------------------------------
# 3. VERIFY DOCUMENTS
# -------------------------------------------------------------------

doc_files = sorted(DOCS_DIR.glob("doc_*.txt"))

print("\nDocuments found:", len(doc_files))

for file in doc_files:
    print("  ✓", file.name)

if len(doc_files) != 8:
    raise RuntimeError("Expected exactly 8 policy documents.")

# -------------------------------------------------------------------
# 4. LOAD EMBEDDING MODEL
# -------------------------------------------------------------------

MODEL_NAME = "all-MiniLM-L6-v2"

print("\nLoading embedding model:", MODEL_NAME)

embedder = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded.")

# -------------------------------------------------------------------
# 5. CREATE CHROMADB
# -------------------------------------------------------------------

print("\nCreating ChromaDB...")

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

COLLECTION_NAME = "zepto_policies"

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# -------------------------------------------------------------------
# 6. EMBED AND STORE ALL DOCUMENTS
# -------------------------------------------------------------------

ids = []
texts = []
metadatas = []

for file in doc_files:

    ids.append(file.stem)

    texts.append(
        file.read_text(encoding="utf-8")
    )

    metadatas.append({
        "document_id": file.stem,
        "source": file.name
    })

print("\nCreating embeddings...")

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True
).tolist()

collection.upsert(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings
)

print("ChromaDB collection:", COLLECTION_NAME)
print("Documents stored:", collection.count())

# -------------------------------------------------------------------
# 7. MOCK LLM CONFIGURATION
# -------------------------------------------------------------------

MOCK_LLM = os.getenv("MOCK_LLM", "1") != "0"

print("\nMOCK_LLM:", MOCK_LLM)

KEYWORDS = (
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours"
)

# -------------------------------------------------------------------
# 8. PYDANTIC RESPONSE MODEL
# -------------------------------------------------------------------

class AskRequest(BaseModel):

    query: str


class AskResponse(BaseModel):

    answer: str

    sources: list[str]

    confidence: float = Field(
        ge=0.0,
        le=1.0
    )

# -------------------------------------------------------------------
# 9. LANGGRAPH STATE
# -------------------------------------------------------------------

class GraphState(TypedDict, total=False):

    query: str

    intent: Literal[
        "policy_question",
        "general_question"
    ]

    retrieved: list[dict[str, Any]]

    answer: str

    sources: list[str]

    confidence: float

# -------------------------------------------------------------------
# 10. INTENT CLASSIFICATION NODE
# -------------------------------------------------------------------

def classify_intent(
    state: GraphState
) -> GraphState:

    query = state["query"]

    if MOCK_LLM:

        query_lower = query.lower()

        if any(
            keyword in query_lower
            for keyword in KEYWORDS
        ):

            intent = "policy_question"

        else:

            intent = "general_question"

        print(
            f"\n[CLASSIFY] {query}"
        )

        print(
            f"[CLASSIFY] Intent = {intent}"
        )

        return {
            "intent": intent
        }

    else:

        return {
            "intent": real_llm_classify(query)
        }

# -------------------------------------------------------------------
# 11. RETRIEVAL FUNCTION
# -------------------------------------------------------------------

def retrieve_policy_chunks(
    query: str,
    n: int = 3
):

    print(
        "\n[RETRIEVAL] Embedding query..."
    )

    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).tolist()[0]

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=n,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved = []

    for i in range(
        len(result["ids"][0])
    ):

        retrieved.append({

            "id": result["ids"][0][i],

            "document":
                result["documents"][0][i],

            "metadata":
                result["metadatas"][0][i],

            "distance":
                float(
                    result["distances"][0][i]
                )

        })

    print(
        "[RETRIEVAL] Retrieved:",
        [item["id"] for item in retrieved]
    )

    return retrieved

# -------------------------------------------------------------------
# 12. STRUCTURED PROMPT
# -------------------------------------------------------------------

PROMPT_TEMPLATE = """

ROLE:
You are a grounded customer-support assistant
answering questions about Zepto's provided policy corpus.

CONTEXT:
Use only the retrieved Zepto policy chunks provided below.

{context}

TASK:
Answer the user's question using the retrieved context.
If the context does not contain enough information,
say that the provided policy context does not specify
the answer.

FORMAT:
Return a JSON object with exactly these fields:

{
    "answer": "string",
    "sources": ["chunk/document ids"],
    "confidence": 0.0
}

The confidence value must be between 0 and 1.

LENGTH:
Keep the answer concise and directly relevant,
preferably 1–3 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information not present in the
provided context. Do not invent or infer Zepto policy
details.

FEW-SHOT EXAMPLE:

User question:
"How long do refunds take?"

Context:
"Approved refunds are credited to the original
payment method within 3–5 business days, or instantly
to the Zepto wallet if the customer opts for wallet credit."

Output:
{
    "answer": "Approved refunds reach the original
    payment method within 3–5 business days, or instantly
    to the Zepto wallet when wallet credit is selected.",
    "sources": ["doc_02"],
    "confidence": 1.0
}

USER QUESTION:
{query}
"""

# -------------------------------------------------------------------
# 13. RETRIEVE AND ANSWER NODE
# -------------------------------------------------------------------

def retrieve_and_answer(
    state: GraphState
) -> GraphState:

    retrieved = retrieve_policy_chunks(
        state["query"],
        n=3
    )

    if MOCK_LLM:

        top_chunk = retrieved[0]

        snippet = top_chunk["document"][:200]

        answer = (
            "Based on the retrieved context: "
            + snippet
        )

        print(
            "\n[GENERATE - MOCK]"
        )

        print(answer)

        return {

            "retrieved": retrieved,

            "answer": answer,

            "sources": [
                item["id"]
                for item in retrieved
            ],

            "confidence": 1.0

        }

    else:

        answer, sources, confidence = (
            real_llm_answer(
                state["query"],
                retrieved
            )
        )

        return {

            "retrieved": retrieved,

            "answer": answer,

            "sources": sources,

            "confidence": confidence

        }

# -------------------------------------------------------------------
# 14. DIRECT ANSWER NODE
# -------------------------------------------------------------------

def direct_answer(
    state: GraphState
) -> GraphState:

    if MOCK_LLM:

        answer = (
            "I can only answer questions "
            "about Zepto policies right now."
        )

        print(
            "\n[DIRECT ANSWER - MOCK]"
        )

        print(answer)

        return {

            "answer": answer,

            "sources": [],

            "confidence": 1.0

        }

    else:

        answer, confidence = (
            real_llm_direct(
                state["query"]
            )
        )

        return {

            "answer": answer,

            "sources": [],

            "confidence": confidence

        }

# -------------------------------------------------------------------
# 15. ROUTER
# -------------------------------------------------------------------

def route_intent(
    state: GraphState
):

    return state["intent"]

# -------------------------------------------------------------------
# 16. OPTIONAL REAL LLM CLIENT
# -------------------------------------------------------------------

LLM_MODEL = os.getenv(
    "LLM_MODEL",
    "gpt-4o-mini"
)

def real_llm_client():

    if OpenAI is None:

        raise RuntimeError(
            "OpenAI package is not installed."
        )

    if not os.getenv(
        "OPENAI_API_KEY"
    ):

        raise RuntimeError(
            "OPENAI_API_KEY is required "
            "when MOCK_LLM=0."
        )

    return OpenAI()

# -------------------------------------------------------------------
# 17. OPTIONAL REAL LLM CLASSIFICATION
# -------------------------------------------------------------------

def real_llm_classify(
    query: str
):

    client = real_llm_client()

    prompt = f"""

Classify this query as exactly one of:

policy_question
general_question

A policy_question needs retrieval from the
Zepto policy corpus.

A general_question does not.

Return only the label.

Query:
{query}
"""

    response = client.chat.completions.create(

        model=LLM_MODEL,

        temperature=0,

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    label = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    if label == "policy_question":

        return "policy_question"

    return "general_question"

# -------------------------------------------------------------------
# 18. JSON VALIDATION
# -------------------------------------------------------------------

def parse_response(
    raw: str
):

    cleaned = raw.strip()

    if cleaned.startswith("```"):

        cleaned = cleaned.strip("`")

        if cleaned.startswith("json"):

            cleaned = cleaned[4:].strip()

    data = json.loads(cleaned)

    return AskResponse.model_validate(data)

# -------------------------------------------------------------------
# 19. OPTIONAL REAL LLM POLICY ANSWER
# -------------------------------------------------------------------

def real_llm_answer(
    query: str,
    retrieved: list[dict[str, Any]]
):

    context = "\n\n".join(

        f"[{item['id']}] {item['document']}"

        for item in retrieved

    )

    prompt = PROMPT_TEMPLATE.format(

        context=context,

        query=query

    )

    client = real_llm_client()

    last_error = None

    for attempt in range(3):

        correction = ""

        if attempt > 0:

            correction = """

Your previous response failed JSON schema
validation.

Return ONLY valid JSON with:

answer
sources
confidence

Confidence must be between 0 and 1.
"""

        response = client.chat.completions.create(

            model=LLM_MODEL,

            temperature=0,

            messages=[
                {
                    "role": "user",
                    "content": prompt + correction
                }
            ]
        )

        raw = (
            response
            .choices[0]
            .message
            .content
            or ""
        )

        try:

            parsed = parse_response(raw)

            return (
                parsed.answer,
                parsed.sources,
                parsed.confidence
            )

        except Exception as exc:

            last_error = exc

    return (
        f"ERROR: real LLM response failed "
        f"schema validation after 3 attempts: "
        f"{last_error}",
        [],
        0.0
    )

# -------------------------------------------------------------------
# 20. OPTIONAL REAL LLM GENERAL ANSWER
# -------------------------------------------------------------------

def real_llm_direct(
    query: str
):

    client = real_llm_client()

    prompt = f"""

ROLE:
You are a customer-support assistant.

CONTEXT:
No Zepto policy retrieval is being used.

TASK:
Answer the general user question directly.

FORMAT:
Return JSON with:

answer
sources
confidence

LENGTH:
Keep the answer concise.

NEGATIVE CONSTRAINT:
Do not claim Zepto policy details because no
policy context was retrieved.

FEW-SHOT EXAMPLE:

Question:
"What is 2+2?"

Output:
{{
    "answer": "4",
    "sources": [],
    "confidence": 1.0
}}

USER QUESTION:
{query}
"""

    last_error = None

    for attempt in range(3):

        correction = ""

        if attempt > 0:

            correction = """
Return ONLY valid JSON containing:
answer, sources, confidence.
"""

        response = client.chat.completions.create(

            model=LLM_MODEL,

            temperature=0,

            messages=[
                {
                    "role": "user",
                    "content": prompt + correction
                }
            ]
        )

        try:

            parsed = parse_response(

                response
                .choices[0]
                .message
                .content
                or ""

            )

            return (
                parsed.answer,
                parsed.confidence
            )

        except Exception as exc:

            last_error = exc

    return (
        f"ERROR: real LLM response failed "
        f"schema validation after 3 attempts: "
        f"{last_error}",
        0.0
    )

# -------------------------------------------------------------------
# 21. BUILD LANGGRAPH
# -------------------------------------------------------------------

print("\nBuilding LangGraph...")

graph_builder = StateGraph(
    GraphState
)

graph_builder.add_node(
    "classify_intent",
    classify_intent
)

graph_builder.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

graph_builder.add_node(
    "direct_answer",
    direct_answer
)

graph_builder.set_entry_point(
    "classify_intent"
)

graph_builder.add_conditional_edges(

    "classify_intent",

    route_intent,

    {
        "policy_question":
            "retrieve_and_answer",

        "general_question":
            "direct_answer"
    }
)

graph_builder.add_edge(
    "retrieve_and_answer",
    END
)

graph_builder.add_edge(
    "direct_answer",
    END
)

graph = graph_builder.compile()

print("LangGraph created successfully.")

# -------------------------------------------------------------------
# 22. TEST 1 - POLICY QUESTION
# -------------------------------------------------------------------

print("\n")
print("=" * 70)
print("TEST 1 - POLICY QUESTION")
print("=" * 70)

query_1 = (
    "What is the delivery fee for orders below INR 149?"
)

result_1 = graph.invoke({
    "query": query_1
})

response_1 = AskResponse(

    answer=result_1["answer"],

    sources=result_1.get(
        "sources",
        []
    ),

    confidence=float(
        result_1.get(
            "confidence",
            1.0
        )
    )
)

print("\nFINAL JSON RESPONSE:")
print(
    json.dumps(
        response_1.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

# -------------------------------------------------------------------
# 23. TEST 2 - GENERAL QUESTION
# -------------------------------------------------------------------

print("\n")
print("=" * 70)
print("TEST 2 - GENERAL QUESTION")
print("=" * 70)

query_2 = (
    "What is the capital of France?"
)

result_2 = graph.invoke({
    "query": query_2
})

response_2 = AskResponse(

    answer=result_2["answer"],

    sources=result_2.get(
        "sources",
        []
    ),

    confidence=float(
        result_2.get(
            "confidence",
            1.0
        )
    )
)

print("\nFINAL JSON RESPONSE:")
print(
    json.dumps(
        response_2.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

# -------------------------------------------------------------------
# 24. TEST 3 - REFUND QUESTION
# -------------------------------------------------------------------

print("\n")
print("=" * 70)
print("TEST 3 - REFUND QUESTION")
print("=" * 70)

query_3 = (
    "How long does a refund take?"
)

result_3 = graph.invoke({
    "query": query_3
})

response_3 = AskResponse(

    answer=result_3["answer"],

    sources=result_3.get(
        "sources",
        []
    ),

    confidence=float(
        result_3.get(
            "confidence",
            1.0
        )
    )
)

print("\nFINAL JSON RESPONSE:")
print(
    json.dumps(
        response_3.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

# -------------------------------------------------------------------
# 25. FINAL STATUS
# -------------------------------------------------------------------

print("\n")
print("=" * 70)
print("RAG PIPELINE TEST COMPLETED")
print("=" * 70)

print("✓ 8 documents created")
print("✓ all-MiniLM-L6-v2 embeddings generated")
print("✓ ChromaDB collection created")
print("✓ ChromaDB document count:", collection.count())
print("✓ LangGraph created")
print("✓ classify_intent node working")
print("✓ retrieve_and_answer node working")
print("✓ direct_answer node working")
print("✓ Conditional routing working")
print("✓ Pydantic validation working")
print("✓ MOCK_LLM baseline working")
print("=" * 70)

In [6]:
from fastapi import FastAPI

app = FastAPI(
    title="Zepto Policy RAG API",
    description="Zepto Policy RAG using LangGraph, ChromaDB and MiniLM",
    version="1.0.0"
)

@app.get("/")
def root():
    return {
        "message": "Zepto Policy RAG API is running"
    }


@app.get("/health")
def health():
    return {
        "status": "ok",
        "mock_llm": MOCK_LLM,
        "collection": COLLECTION_NAME,
        "document_count": collection.count()
    }


@app.post("/ask", response_model=AskResponse)
def ask(request: AskRequest):

    result = graph.invoke({
        "query": request.query
    })

    response = AskResponse(
        answer=result["answer"],
        sources=result.get("sources", []),
        confidence=float(
            result.get("confidence", 1.0)
        )
    )

    return response


print("FastAPI application created successfully.")

FastAPI application created successfully.


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

response = client.post(
    "/ask",
    json={
        "query": "What is the delivery fee for orders below INR 149?"
    }
)

print("Status code:", response.status_code)

print("\nResponse:")
print(response.json())

In [ ]:
response = client.post(
    "/ask",
    json={
        "query": "What is the capital of France?"
    }
)

print("Status code:", response.status_code)

print("\nResponse:")
print(response.json())

response = client.get("/health")

print(response.json())



In [8]:
from pathlib import Path

project_dir = Path.cwd()

print("Project directory:")
print(project_dir)

Project directory:
c:\Users\krish\OneDrive\Desktop\Python AI ML


In [9]:
main_file = project_dir / "main.py"

main_file.write_text(
    '''# Zepto Policy RAG
# Generated from the working Jupyter implementation.

import os
import json
from pathlib import Path
from typing import TypedDict, Literal, Any

import chromadb
from fastapi import FastAPI
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

BASE_DIR = Path(__file__).resolve().parent
DOCS_DIR = BASE_DIR / "docs"
CHROMA_DIR = BASE_DIR / "chroma_db"

COLLECTION_NAME = "zepto_policies"
MODEL_NAME = "all-MiniLM-L6-v2"

MOCK_LLM = os.getenv("MOCK_LLM", "1") != "0"

KEYWORDS = (
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours"
)


class AskRequest(BaseModel):
    query: str


class AskResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(
        ge=0.0,
        le=1.0
    )


class GraphState(TypedDict, total=False):
    query: str
    intent: Literal[
        "policy_question",
        "general_question"
    ]
    retrieved: list[dict[str, Any]]
    answer: str
    sources: list[str]
    confidence: float


embedder = SentenceTransformer(MODEL_NAME)

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)


def ingest_documents():

    ids = []
    documents = []
    metadatas = []

    for path in sorted(
        DOCS_DIR.glob("doc_*.txt")
    ):

        ids.append(path.stem)

        documents.append(
            path.read_text(
                encoding="utf-8"
            )
        )

        metadatas.append({
            "document_id": path.stem,
            "source": path.name
        })

    if not ids:
        raise RuntimeError(
            "No policy documents found."
        )

    embeddings = embedder.encode(
        documents,
        normalize_embeddings=True
    ).tolist()

    collection.upsert(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )


ingest_documents()


def classify_intent(
    state: GraphState
):

    query = state["query"]

    if MOCK_LLM:

        query_lower = query.lower()

        if any(
            keyword in query_lower
            for keyword in KEYWORDS
        ):
            intent = "policy_question"
        else:
            intent = "general_question"

    else:

        intent = real_llm_classify(
            query
        )

    return {
        "intent": intent
    }


def retrieve_policy_chunks(
    query: str,
    n: int = 3
):

    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).tolist()[0]

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=n,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved = []

    for i in range(
        len(result["ids"][0])
    ):

        retrieved.append({

            "id": result["ids"][0][i],

            "document":
                result["documents"][0][i],

            "metadata":
                result["metadatas"][0][i],

            "distance":
                float(
                    result["distances"][0][i]
                )
        })

    return retrieved


PROMPT_TEMPLATE = """
ROLE:
You are a grounded customer-support assistant
answering questions about Zepto's provided policy corpus.

CONTEXT:
Use only the retrieved Zepto policy chunks provided below.

{context}

TASK:
Answer the user's question using the retrieved context.
If the context does not contain enough information,
say that the provided policy context does not specify
the answer.

FORMAT:
Return JSON with:
answer, sources, confidence.

LENGTH:
Keep the answer concise and directly relevant.

NEGATIVE CONSTRAINT:
Do not answer using information not present in the
provided context. Do not invent or infer Zepto policy details.

FEW-SHOT EXAMPLE:

User question:
"How long do refunds take?"

Context:
"Approved refunds are credited to the original payment
method within 3–5 business days, or instantly to the
Zepto wallet if the customer opts for wallet credit."

Output:
{
    "answer": "Approved refunds reach the original payment
    method within 3–5 business days, or instantly to the
    Zepto wallet when wallet credit is selected.",
    "sources": ["doc_02"],
    "confidence": 1.0
}

USER QUESTION:
{query}
"""


def retrieve_and_answer(
    state: GraphState
):

    retrieved = retrieve_policy_chunks(
        state["query"],
        n=3
    )

    if MOCK_LLM:

        top_chunk = retrieved[0]

        snippet = top_chunk[
            "document"
        ][:200]

        answer = (
            "Based on the retrieved context: "
            + snippet
        )

        return {

            "retrieved": retrieved,

            "answer": answer,

            "sources": [
                item["id"]
                for item in retrieved
            ],

            "confidence": 1.0
        }

    else:

        answer, sources, confidence = (
            real_llm_answer(
                state["query"],
                retrieved
            )
        )

        return {

            "retrieved": retrieved,

            "answer": answer,

            "sources": sources,

            "confidence": confidence
        }


def direct_answer(
    state: GraphState
):

    if MOCK_LLM:

        return {

            "answer":
                "I can only answer questions "
                "about Zepto policies right now.",

            "sources": [],

            "confidence": 1.0
        }

    else:

        answer, confidence = (
            real_llm_direct(
                state["query"]
            )
        )

        return {

            "answer": answer,

            "sources": [],

            "confidence": confidence
        }


def route_intent(
    state: GraphState
):

    return state["intent"]


def parse_response(
    raw: str
):

    cleaned = raw.strip()

    if cleaned.startswith("```"):

        cleaned = cleaned.strip("`")

        if cleaned.startswith("json"):

            cleaned = cleaned[4:].strip()

    data = json.loads(cleaned)

    return AskResponse.model_validate(
        data
    )


def real_llm_client():

    if OpenAI is None:

        raise RuntimeError(
            "Install openai package."
        )

    if not os.getenv(
        "OPENAI_API_KEY"
    ):

        raise RuntimeError(
            "OPENAI_API_KEY is required."
        )

    return OpenAI()


def real_llm_classify(
    query: str
):

    client = real_llm_client()

    prompt = f"""
Classify this query as exactly:
policy_question
or
general_question

Query:
{query}

Return only the label.
"""

    response = client.chat.completions.create(

        model=os.getenv(
            "LLM_MODEL",
            "gpt-4o-mini"
        ),

        temperature=0,

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    label = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

    if label == "policy_question":

        return "policy_question"

    return "general_question"


def real_llm_answer(
    query: str,
    retrieved: list[dict[str, Any]]
):

    context = "\\n\\n".join(

        f"[{item['id']}] {item['document']}"

        for item in retrieved

    )

    prompt = PROMPT_TEMPLATE.format(
        context=context,
        query=query
    )

    client = real_llm_client()

    last_error = None

    for attempt in range(3):

        correction = ""

        if attempt > 0:

            correction = """
Previous response failed validation.
Return ONLY valid JSON with:
answer, sources, confidence.
"""

        response = client.chat.completions.create(

            model=os.getenv(
                "LLM_MODEL",
                "gpt-4o-mini"
            ),

            temperature=0,

            messages=[
                {
                    "role": "user",
                    "content":
                        prompt + correction
                }
            ]
        )

        try:

            parsed = parse_response(
                response
                .choices[0]
                .message
                .content
            )

            return (
                parsed.answer,
                parsed.sources,
                parsed.confidence
            )

        except Exception as exc:

            last_error = exc

    return (
        f"ERROR: validation failed: {last_error}",
        [],
        0.0
    )


def real_llm_direct(
    query: str
):

    client = real_llm_client()

    prompt = f"""
Answer this general question.

Do not invent Zepto policy details.

Return JSON with:
answer, sources, confidence.

Question:
{query}
"""

    for attempt in range(3):

        try:

            response = client.chat.completions.create(

                model=os.getenv(
                    "LLM_MODEL",
                    "gpt-4o-mini"
                ),

                temperature=0,

                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            parsed = parse_response(
                response
                .choices[0]
                .message
                .content
            )

            return (
                parsed.answer,
                parsed.confidence
            )

        except Exception:
            continue

    return (
        "ERROR: validation failed",
        0.0
    )


graph_builder = StateGraph(
    GraphState
)

graph_builder.add_node(
    "classify_intent",
    classify_intent
)

graph_builder.add_node(
    "retrieve_and_answer",
    retrieve_and_answer
)

graph_builder.add_node(
    "direct_answer",
    direct_answer
)

graph_builder.set_entry_point(
    "classify_intent"
)

graph_builder.add_conditional_edges(

    "classify_intent",

    route_intent,

    {
        "policy_question":
            "retrieve_and_answer",

        "general_question":
            "direct_answer"
    }
)

graph_builder.add_edge(
    "retrieve_and_answer",
    END
)

graph_builder.add_edge(
    "direct_answer",
    END
)

graph = graph_builder.compile()


app = FastAPI(
    title="Zepto Policy RAG API",
    version="1.0.0"
)


@app.get("/")
def root():

    return {
        "message":
            "Zepto Policy RAG API is running"
    }


@app.get("/health")
def health():

    return {

        "status": "ok",

        "mock_llm": MOCK_LLM,

        "collection":
            COLLECTION_NAME,

        "document_count":
            collection.count()
    }


@app.post(
    "/ask",
    response_model=AskResponse
)
def ask(
    request: AskRequest
):

    result = graph.invoke({

        "query":
            request.query

    })

    return AskResponse(

        answer=result["answer"],

        sources=result.get(
            "sources",
            []
        ),

        confidence=float(
            result.get(
                "confidence",
                1.0
            )
        )
    )
''',
    encoding="utf-8"
)

print("main.py created:")
print(main_file)

main.py created:
c:\Users\krish\OneDrive\Desktop\Python AI ML\main.py


In [10]:
requirements = """fastapi>=0.115,<1
uvicorn[standard]>=0.30,<1
chromadb>=1.0,<2
sentence-transformers>=3.0,<6
langgraph>=0.2,<2
pydantic>=2.7,<3
openai>=1.40,<2
"""

(project_dir / "requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)

print("requirements.txt created")

requirements.txt created


In [11]:
dockerfile = """FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV MOCK_LLM=1

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY docs ./docs
COPY main.py .

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]
"""

(project_dir / "Dockerfile").write_text(
    dockerfile,
    encoding="utf-8"
)

print("Dockerfile created")

Dockerfile created


In [12]:
dockerignore = """__pycache__/
*.pyc
.ipynb_checkpoints/
.venv/
.git/
chroma_db/
"""

(project_dir / ".dockerignore").write_text(
    dockerignore,
    encoding="utf-8"
)

print(".dockerignore created")

.dockerignore created


In [13]:
for item in sorted(project_dir.iterdir()):
    print(item.name)

.dockerignore
.git
analytics.ipynb
books_database.db
chroma_db
datapipeline.ipynb
Dockerfile
docs
main.py
requirements.txt
supportassisstant.ipynb
tiatanic.csv
titanic_best_pipeline.joblib
venv
